In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import copy
import math

## Scenario 1: AORC data is used for both training and prediction

#### US-UC1 & US-UC2

In [19]:
AORC_Rider18 = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'Rider_Rider 18_combined_aorc_data.csv')).drop(['latitude', 'longitude'], axis=1)

AORC_Rider18['time'] = pd.to_datetime(AORC_Rider18['time'])
AORC_Rider18.set_index('time', inplace=True)
AORC_Rider18.index = AORC_Rider18.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [
    ('2019-05-20', '2019-09-11'),
    ('2020-05-27', '2020-09-25'),
    ('2021-05-20', '2021-11-16'),
    ('2022-06-08', '2022-12-08')
]

new_growing_seasons = [
    ('2019-04-01', '2019-10-01'),
    ('2020-05-01', '2020-10-15'),
    ('2021-04-01', '2021-12-01'),
    ('2022-05-01', '2022-12-30')
]


AORC_Rider18_filtered = pd.concat([AORC_Rider18.loc[start:end] for (start, end) in new_growing_seasons])

# Columns convertion
AORC_Rider18_filtered['Air Temperature'] = AORC_Rider18_filtered['Air Temperature'] - 273.15

AORC_Rider18_filtered['Wind Speed'] = np.sqrt(AORC_Rider18_filtered['U-Component of Wind']**2 + AORC_Rider18_filtered['V-Component of Wind']**2)

AORC_Rider18_filtered['Air Pressure'] = AORC_Rider18_filtered['Pressure']/1000

AORC_Rider18_filtered = AORC_Rider18_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Pressure' ], axis=1, inplace=False)

AORC_Rider18_filtered =AORC_Rider18_filtered[['Total Precipitation', 'Air Temperature','Specific Humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Air Pressure', 'Wind Speed']]

# add some other features to capture prioir conditions in the predictions as well
AORC_Rider18_filtered['Total_Precipitation_12h_sum'] = AORC_Rider18_filtered['Total Precipitation'].rolling(window='12H').sum()
AORC_Rider18_filtered['Total_Precipitation_3h_sum'] = AORC_Rider18_filtered['Total Precipitation'].rolling(window='3H').sum()
AORC_Rider18_filtered['Specific_Humidity_3h_mean'] = AORC_Rider18_filtered['Specific Humidity'].rolling(window='3H').mean()

# this would make the data to match the tower data hourly aggregation
AORC_Rider18_filtered.index = AORC_Rider18_filtered.index - pd.DateOffset(hours=0)

AORC_Rider18_filtered = AORC_Rider18_filtered.between_time('6:00', '18:00')

In [20]:
AORC_Rider18_filtered.to_csv('US_UC1_UC2_2019-2022_filtered_7_vars_AORC.csv')

#### US-HWB

In [21]:
AORC_HWB = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'US-HWB_combined_aorc_data.csv')).drop(['latitude', 'longitude'], axis=1)

AORC_HWB['time'] = pd.to_datetime(AORC_HWB['time'])
AORC_HWB.set_index('time', inplace=True)

AORC_HWB.index = AORC_HWB.index - pd.DateOffset(hours=6)

# Define the time windows
growing_seasons = [
    ('2017-04-15', '2017-09-30'),
]


AORC_HWB_filtered = pd.concat([AORC_HWB.loc[start:end] for (start, end) in growing_seasons])

# Columns convertion
AORC_HWB_filtered['Air Temperature'] = AORC_HWB_filtered['Air Temperature'] - 273.15

AORC_HWB_filtered['Wind Speed'] = np.sqrt(AORC_HWB_filtered['U-Component of Wind']**2 + AORC_HWB_filtered['V-Component of Wind']**2)

AORC_HWB_filtered['Air Pressure'] = AORC_HWB_filtered['Pressure']/1000

AORC_HWB_filtered = AORC_HWB_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Pressure' ], axis=1, inplace=False)

AORC_HWB_filtered =AORC_HWB_filtered[['Total Precipitation', 'Air Temperature','Specific Humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Air Pressure', 'Wind Speed']]

# add some other features to capture prioir conditions in the predictions as well
AORC_HWB_filtered['Total_Precipitation_12h_sum'] = AORC_HWB_filtered['Total Precipitation'].rolling(window='12H').sum()
AORC_HWB_filtered['Total_Precipitation_3h_sum'] = AORC_HWB_filtered['Total Precipitation'].rolling(window='3H').sum()
AORC_HWB_filtered['Specific_Humidity_3h_mean'] = AORC_HWB_filtered['Specific Humidity'].rolling(window='3H').mean()

# this would make the data to match the tower data hourly aggregation
AORC_HWB_filtered.index = AORC_HWB_filtered.index - pd.DateOffset(hours=0)

AORC_HWB_filtered = AORC_HWB_filtered.between_time('6:00', '18:00')

In [22]:
AORC_HWB_filtered.to_csv('US_HWB_2017_filtered_7_vars_AORC.csv')

## Scenario 2: AORC data is only used for prediction (training with tower met variables)

In [2]:
def specific_to_relative_humidity(specific_humidity, temperature, pressure):
    """
    Convert specific humidity to relative humidity.

    Parameters:
    specific_humidity (float): Specific humidity in kg/kg.
    temperature (float): Temperature in degrees Celsius.
    pressure (float): Atmospheric pressure in kPa.

    Returns:
    float: Relative humidity in percentage.
    """

    # Calculate the actual vapor pressure (e)
    e = (specific_humidity * pressure *10 ) / (0.622 + 0.378 * specific_humidity)

    # Calculate the saturation vapor pressure (es) using Tetens' formula
    es = 6.112 * math.exp((17.67 * temperature) / (temperature + 243.5))

    # Calculate relative humidity (RH)
    relative_humidity = (e / es) * 100

    return relative_humidity

#### US-UC1 & US-UC2

In [ ]:
AORC_Rider18 = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'Rider_Rider 18_combined_aorc_data.csv')).drop(['latitude', 'longitude'], axis=1)

AORC_Rider18['time'] = pd.to_datetime(AORC_Rider18['time'])
AORC_Rider18.set_index('time', inplace=True)
AORC_Rider18.index = AORC_Rider18.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [
    ('2019-05-20', '2019-09-11'),
    ('2020-05-27', '2020-09-25'),
    ('2021-05-20', '2021-11-16'),
    ('2022-06-08', '2022-12-08')
]

AORC_Rider18_filtered = pd.concat([AORC_Rider18.loc[start:end] for (start, end) in growing_seasons]).between_time('5:00', '19:00')

# Columns convertion
AORC_Rider18_filtered['Air Temperature'] = AORC_Rider18_filtered['Air Temperature'] - 273.15

AORC_Rider18_filtered['Wind Speed'] = np.sqrt(AORC_Rider18_filtered['U-Component of Wind']**2 + AORC_Rider18_filtered['V-Component of Wind']**2)

AORC_Rider18_filtered['Pressure'] = AORC_Rider18_filtered['Pressure']/1000

AORC_Rider18_filtered['relative_humidity'] = AORC_Rider18_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_Rider18_filtered = AORC_Rider18_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_Rider18_filtered =AORC_Rider18_filtered[['Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]

# this would make the data to match the tower data hourly aggregation
AORC_Rider18_filtered.index = AORC_Rider18_filtered.index - pd.DateOffset(hours=0)

AORC_Rider18_filtered = AORC_Rider18_filtered.between_time('6:00', '18:00')

In [9]:
AORC_Rider18_filtered

,Total Precipitation,Air Temperature,relative_humidity,Downward Long-Wave Radiation Flux,Downward Short-Wave Radiation Flux,Pressure,Wind Speed
time,,,,,,,
2019-05-20 10:00:00,0.0,15.45,97.903732,383.4,5.90000,96.02,0.984886
2019-05-20 11:00:00,0.0,16.55,99.357428,383.4,93.90000,96.03,1.220656
2019-05-20 12:00:00,0.0,17.75,95.072611,358.9,258.20000,96.02,0.921954
2019-05-20 13:00:00,0.0,19.25,90.629297,358.9,391.70000,96.02,1.565248
2019-05-20 14:00:00,0.0,19.85,84.693803,358.9,515.20000,96.03,1.843909
...,...,...,...,...,...,...,...
2022-12-08 18:00:00,0.0,7.35,65.494279,292.1,324.20000,97.43,1.897367
2022-12-08 19:00:00,0.0,7.25,65.876137,292.1,265.30002,97.33,1.811077
2022-12-08 20:00:00,0.0,7.05,65.242190,292.1,185.40001,97.34,1.992486


In [9]:
AORC_Rider18_filtered.to_csv('US_UC1_UC2_2019-2022_filtered_7_vars_AORC_MOD.csv')

#### 2023 

In [10]:
AORC_Rider18 = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'Rider_Rider 18_combined_aorc_data.csv')).drop(['latitude', 'longitude'], axis=1)

AORC_Rider18['time'] = pd.to_datetime(AORC_Rider18['time'])
AORC_Rider18.set_index('time', inplace=True)
AORC_Rider18.index = AORC_Rider18.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [
    ('2023-06-02', '2023-10-26')
]

AORC_Rider18_filtered = pd.concat([AORC_Rider18.loc[start:end] for (start, end) in growing_seasons]).between_time('5:00', '19:00')

# Columns convertion
AORC_Rider18_filtered['Air Temperature'] = AORC_Rider18_filtered['Air Temperature'] - 273.15

AORC_Rider18_filtered['Wind Speed'] = np.sqrt(AORC_Rider18_filtered['U-Component of Wind']**2 + AORC_Rider18_filtered['V-Component of Wind']**2)

AORC_Rider18_filtered['Pressure'] = AORC_Rider18_filtered['Pressure']/1000

AORC_Rider18_filtered['relative_humidity'] = AORC_Rider18_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_Rider18_filtered = AORC_Rider18_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_Rider18_filtered =AORC_Rider18_filtered[['Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]

# this would make the data to match the tower data hourly aggregation
AORC_Rider18_filtered.index = AORC_Rider18_filtered.index - pd.DateOffset(hours=0)

AORC_Rider18_2023_filtered = AORC_Rider18_filtered.between_time('6:00', '18:00')

In [12]:
AORC_Rider18_2023_filtered.to_csv('US_UC1_UC2_2023_filtered_7_vars_AORC_MOD.csv')

#### US-UC1 & US-UC2 2024

In [3]:
AORC_Rider18_2024 = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'Rider_Rider_18_Full 2024 AORC data.csv')).drop(['latitude', 'longitude'], axis=1)

AORC_Rider18_2024['time'] = pd.to_datetime(AORC_Rider18_2024['time'])
AORC_Rider18_2024.set_index('time', inplace=True)
AORC_Rider18_2024.index = AORC_Rider18_2024.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [('2024-05-25', '2024-10-08')]

AORC_Rider18_2024_filtered = pd.concat([AORC_Rider18_2024.loc[start:end] for (start, end) in growing_seasons]).between_time('5:00', '19:00')

# Columns convertion
AORC_Rider18_2024_filtered['Air Temperature'] = AORC_Rider18_2024_filtered['Air Temperature'] - 273.15

AORC_Rider18_2024_filtered['Wind Speed'] = np.sqrt(AORC_Rider18_2024_filtered['U-Component of Wind']**2 + AORC_Rider18_2024_filtered['V-Component of Wind']**2)

AORC_Rider18_2024_filtered['Pressure'] = AORC_Rider18_2024_filtered['Pressure']/1000

AORC_Rider18_2024_filtered['relative_humidity'] = AORC_Rider18_2024_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_Rider18_2024_filtered = AORC_Rider18_2024_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_Rider18_2024_filtered =AORC_Rider18_2024_filtered[['Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]

# this would make the data to match the tower data hourly aggregation
AORC_Rider18_2024_filtered.index = AORC_Rider18_2024_filtered.index - pd.DateOffset(hours=0)

AORC_Rider18_2024_filtered = AORC_Rider18_2024_filtered.between_time('6:00', '18:00')

In [4]:
AORC_Rider18_2024_filtered.to_csv('US_UC1_UC2_2024_filtered_7_vars_AORC_MOD.csv')

#### HWB

In [5]:
AORC_HWB = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'US-HWB_combined_aorc_data.csv')).drop(['latitude', 'longitude'], axis=1)

AORC_HWB['time'] = pd.to_datetime(AORC_HWB['time'])
AORC_HWB.set_index('time', inplace=True)
AORC_HWB.index = AORC_HWB.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [('2017-04-15', '2017-09-30')]

AORC_HWB_filtered = pd.concat([AORC_HWB.loc[start:end] for (start, end) in growing_seasons]).between_time('5:00', '19:00')

# Columns convertion
AORC_HWB_filtered['Air Temperature'] = AORC_HWB_filtered['Air Temperature'] - 273.15

AORC_HWB_filtered['Wind Speed'] = np.sqrt(AORC_HWB_filtered['U-Component of Wind']**2 + AORC_HWB_filtered['V-Component of Wind']**2)

AORC_HWB_filtered['Pressure'] = AORC_HWB_filtered['Pressure']/1000

AORC_HWB_filtered['relative_humidity'] = AORC_HWB_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_HWB_filtered = AORC_HWB_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_HWB_filtered =AORC_HWB_filtered[['Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]

# this would make the data to match the tower data hourly aggregation
AORC_HWB_filtered.index = AORC_HWB_filtered.index - pd.DateOffset(hours=0)

AORC_HWB_filtered = AORC_HWB_filtered.between_time('6:00', '18:00')

In [6]:
AORC_HWB_filtered.to_csv('US_HWB_2017_filtered_7_vars_AORC_MOD.csv')

## CA

#### Bi1

In [23]:
AORC_df = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'Bi1_2018_2024_AORC.csv'))

time_utc = pd.to_datetime(AORC_df['time'], utc=True)
AORC_df["time_local"] = time_utc.dt.tz_convert("America/Los_Angeles")
AORC_df.set_index('time_local', inplace=True)
#AORC_df.index = AORC_df.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [
    ('2018-02-05', '2018-10-25'),
    ('2019-02-05', '2019-10-25'),
    ('2020-02-05', '2020-10-25'),
    ('2021-02-05', '2021-10-25'),
    ('2022-02-05', '2022-10-25'),
    ('2023-02-05', '2023-10-25'),
    ('2024-02-05', '2024-09-25')
]

AORC_df_filtered = pd.concat([AORC_df.loc[start:end] for (start, end) in growing_seasons])

# Columns convertion
AORC_df_filtered['Air Temperature'] = AORC_df_filtered['Air Temperature'] - 273.15

AORC_df_filtered['Wind Speed'] = np.sqrt(AORC_df_filtered['U-Component of Wind']**2 + AORC_df_filtered['V-Component of Wind']**2)

AORC_df_filtered['Pressure'] = AORC_df_filtered['Pressure']/1000

AORC_df_filtered['relative_humidity'] = AORC_df_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_df_filtered = AORC_df_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_df_filtered =AORC_df_filtered[['latitude', 'longitude','Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]


AORC_df_filtered = AORC_df_filtered.between_time('6:00', '18:00')

In [24]:
AORC_df_filtered.to_csv('US_Bi1_2018-2024_filtered_7_vars_AORC_MOD.csv')

#### Bi2

In [25]:
AORC_df = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'Bi2_2018_2024_AORC.csv'))

time_utc = pd.to_datetime(AORC_df['time'], utc=True)
AORC_df["time_local"] = time_utc.dt.tz_convert("America/Los_Angeles")
AORC_df.set_index('time_local', inplace=True)
#AORC_df.index = AORC_df.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [
    ('2018-05-26', '2018-09-22'),
    ('2019-05-25', '2019-09-14'),
    ('2020-04-30', '2020-10-12'),
    ('2021-05-03', '2021-10-13'),
    ('2022-05-07', '2022-10-13'),
    ('2023-06-01', '2023-10-27'),
    ('2024-05-09', '2024-10-10')
]

AORC_df_filtered = pd.concat([AORC_df.loc[start:end] for (start, end) in growing_seasons])

# Columns convertion
AORC_df_filtered['Air Temperature'] = AORC_df_filtered['Air Temperature'] - 273.15

AORC_df_filtered['Wind Speed'] = np.sqrt(AORC_df_filtered['U-Component of Wind']**2 + AORC_df_filtered['V-Component of Wind']**2)

AORC_df_filtered['Pressure'] = AORC_df_filtered['Pressure']/1000

AORC_df_filtered['relative_humidity'] = AORC_df_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_df_filtered = AORC_df_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_df_filtered =AORC_df_filtered[['latitude', 'longitude','Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]


AORC_df_filtered = AORC_df_filtered.between_time('6:00', '18:00')

In [26]:
AORC_df_filtered.to_csv('US_Bi2_2018-2024_filtered_7_vars_AORC_MOD.csv')

#### Tw3

In [27]:
AORC_df = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'Tw3_2017_2018_AORC.csv'))

time_utc = pd.to_datetime(AORC_df['time'], utc=True)
AORC_df["time_local"] = time_utc.dt.tz_convert("America/Los_Angeles")
AORC_df.set_index('time_local', inplace=True)
#AORC_df.index = AORC_df.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [
    ('2017-02-05', '2017-09-26'),
    ('2018-02-05', '2018-06-01')]

AORC_df_filtered = pd.concat([AORC_df.loc[start:end] for (start, end) in growing_seasons])

# Columns convertion
AORC_df_filtered['Air Temperature'] = AORC_df_filtered['Air Temperature'] - 273.15

AORC_df_filtered['Wind Speed'] = np.sqrt(AORC_df_filtered['U-Component of Wind']**2 + AORC_df_filtered['V-Component of Wind']**2)

AORC_df_filtered['Pressure'] = AORC_df_filtered['Pressure']/1000

AORC_df_filtered['relative_humidity'] = AORC_df_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_df_filtered = AORC_df_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_df_filtered =AORC_df_filtered[['latitude', 'longitude','Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]

AORC_df_filtered.to_csv('US_Tw3_2017-2018_filtered_7_vars_AORC_MOD.csv')

## IL

In [29]:
AORC_df = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'UiABC_2017_2024_AORC.csv'))

time_utc = pd.to_datetime(AORC_df['time'], utc=True)
AORC_df["time_local"] = time_utc.dt.tz_convert("America/Chicago")
AORC_df.set_index('time_local', inplace=True)
#AORC_df.index = AORC_df.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [
    ('2017-05-15', '2017-09-22'),
    ('2018-05-26', '2018-09-22'),
    ('2019-05-25', '2019-09-14'),
    ('2020-04-30', '2020-10-12'),
    ('2021-05-03', '2021-10-13'),
    ('2022-05-07', '2022-10-13'),
    ('2023-06-01', '2023-10-27'),
    ('2024-05-09', '2024-10-10')
]

AORC_df_filtered = pd.concat([AORC_df.loc[start:end] for (start, end) in growing_seasons])

# Columns convertion
AORC_df_filtered['Air Temperature'] = AORC_df_filtered['Air Temperature'] - 273.15

AORC_df_filtered['Wind Speed'] = np.sqrt(AORC_df_filtered['U-Component of Wind']**2 + AORC_df_filtered['V-Component of Wind']**2)

AORC_df_filtered['Pressure'] = AORC_df_filtered['Pressure']/1000

AORC_df_filtered['relative_humidity'] = AORC_df_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_df_filtered = AORC_df_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_df_filtered =AORC_df_filtered[['latitude', 'longitude','Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]

AORC_df_filtered.to_csv('US_UiABC_2017-2024_filtered_7_vars_AORC_MOD.csv')

## IN

In [31]:
AORC_df = pd.read_csv(os.path.join(os.getcwd(), 'Data', 'VT12_2023_2024_AORC.csv'))

time_utc = pd.to_datetime(AORC_df['time'], utc=True)
AORC_df["time_local"] = time_utc.dt.tz_convert("America/New_York")
AORC_df.set_index('time_local', inplace=True)
#AORC_df.index = AORC_df.index - pd.DateOffset(hours=5)

# Define the time windows
growing_seasons = [('2023-06-01', '2023-10-27'),
    ('2024-05-09', '2024-10-10')
]

AORC_df_filtered = pd.concat([AORC_df.loc[start:end] for (start, end) in growing_seasons])

# Columns convertion
AORC_df_filtered['Air Temperature'] = AORC_df_filtered['Air Temperature'] - 273.15

AORC_df_filtered['Wind Speed'] = np.sqrt(AORC_df_filtered['U-Component of Wind']**2 + AORC_df_filtered['V-Component of Wind']**2)

AORC_df_filtered['Pressure'] = AORC_df_filtered['Pressure']/1000

AORC_df_filtered['relative_humidity'] = AORC_df_filtered.apply(
    lambda row: specific_to_relative_humidity(row['Specific Humidity'], row['Air Temperature'], row['Pressure']),
    axis=1
)

AORC_df_filtered = AORC_df_filtered.drop(['U-Component of Wind', 'V-Component of Wind', 'Specific Humidity'], axis=1, inplace=False)

AORC_df_filtered =AORC_df_filtered[['latitude', 'longitude','Total Precipitation', 'Air Temperature','relative_humidity', 
                                              'Downward Long-Wave Radiation Flux', 'Downward Short-Wave Radiation Flux', 'Pressure', 'Wind Speed']]

AORC_df_filtered.to_csv('US_VT12_2023-2024_filtered_7_vars_AORC_MOD.csv')